# Tier 1 — Grid Search + 5-fold CV (LSSVMs + XGBoost)

Protocolo do orientador (Marinho et al.) aplicado a 12 modelos × 9 datasets × 30 seeds:
- Hold-out 70/30 estratificado por seed
- `GridSearchCV(cv=5, scoring='f1_macro', refit=True)` no $C_{train}$
- Uma única medida em $C_{test}$ por seed

**Modelos (CPU, sem GPU):**
- LSSVM baselines: Standard, PCP, FSA, IP, Pruning, OppositeMaps
- Paper-base + propostos: ADMM-N, ADMM-EN, FISTA-N, DualFISTA, Nyström
- Baseline geral: XGBoost

**Antes de rodar:** `Runtime → Change runtime type → CPU`. **Colab Pro recomenda High-RAM (8 cores)**.

**Resume robusto:** restaura do Drive ao começar, sync em background a cada 5 min, save final ao terminar. Se a sessão cair, basta reabrir e rodar tudo — restaura automaticamente.

**Notebook complementar:** `tier1_gridcv_transformers_colab.ipynb` (FT-Transformers no T4).

In [ ]:
# ── Célula 1: Verifica runtime ──────────────────────────────────────────────
import platform, multiprocessing
print(f'CPU: {platform.processor() or "x86_64"}')
print(f'Cores: {multiprocessing.cpu_count()}')
if multiprocessing.cpu_count() < 4:
    print('\n⚠️  Free tier (2 cores). Considere Colab Pro High-RAM (8 cores).')

In [ ]:
# ── Célula 2: Clonar do GitHub ──────────────────────────────────────────────
import os
PROJECT_DIR = '/content/sparse-lssvm-transformers-study'
GIT_URL = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'

if os.path.exists(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull --rebase
else:
    !git clone {GIT_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
!git log --oneline -3
print(f'\nDiretório atual: {os.getcwd()}')

In [ ]:
# ── Célula 3: Dependências (sem torch) ──────────────────────────────────────
!pip install -q numpy scipy scikit-learn pandas xgboost xlrd pyarrow

import numpy, scipy, sklearn, xgboost
print(f'numpy {numpy.__version__} | scipy {scipy.__version__} | '
      f'sklearn {sklearn.__version__} | xgboost {xgboost.__version__}')

In [ ]:
# ── Célula 4: Baixar datasets Tier 1 ────────────────────────────────────────
!python scripts/download_data.py --tier 1
!ls -lh data/raw/ | head -15

In [ ]:
# ── Célula 5: Montar Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/dissertacao_tier1_gridcv'
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f'Drive em: {DRIVE_PATH}')
!ls -lh '{DRIVE_PATH}' 2>/dev/null

In [ ]:
# ── Célula 6: Restaurar progresso (resume do Drive) ─────────────────────────
import shutil
from pathlib import Path

OUTPUT_FNAME = 'tier1_gridcv_lssvm.json'
drive_results = Path(DRIVE_PATH)
local_results = Path('results')
local_results.mkdir(exist_ok=True)

src = drive_results / OUTPUT_FNAME
dst = local_results / OUTPUT_FNAME
if src.exists():
    shutil.copy(src, dst)
    import json
    n = len(json.load(open(dst)))
    print(f'✓ Restaurado: {OUTPUT_FNAME} ({n} entries)')
else:
    print(f'• Começando do zero: {OUTPUT_FNAME}')

In [ ]:
# ── Célula 7: Sync para Drive em background (a cada 5 min) ─────────────────
%%writefile /content/sync_to_drive.sh
#!/bin/bash
while true; do
    sleep 300
    cp -u /content/sparse-lssvm-transformers-study/results/tier1_gridcv_lssvm.json \
          "$1/tier1_gridcv_lssvm.json" 2>/dev/null
done

In [ ]:
import subprocess
sync_proc = subprocess.Popen(['bash', '/content/sync_to_drive.sh', DRIVE_PATH])
print(f'Sync rodando em background (PID {sync_proc.pid}) — salva a cada 5 min')

In [ ]:
# ── Célula 8: Inspecionar a grade que será rodada ───────────────────────────
from src.tuning.grids import GRIDS, grid_size

LSSVM_MODELS = [
    'StandardLSSVM', 'PCPLSSVm', 'FSALSSVm', 'IPLSSVm',
    'PruningLSSVM', 'OppositeMapsLSSVM',
    'ADMMNesterovLSSVM', 'ADMMElasticNet',
    'FISTANesterov', 'DualFISTA',
    'NystromLSSVMColnorm',
    'XGBoost',
]

print(f'{"Modelo":<25}{"Pontos":>8}{"Fits/seed/dataset":>20}')
print('-' * 55)
for m in LSSVM_MODELS:
    g = grid_size(m)
    print(f'{m:<25}{g:>8}{g*5:>20}')
print(f'\nTotal: 12 modelos × 9 datasets × 30 seeds = 3240 entries')

In [ ]:
# ── Célula 9: Rodar tudo ────────────────────────────────────────────────────
# Script escreve em results/ (local SSD, rápido). O sync da Célula 7 copia pro Drive a cada 5 min.
# Se a sessão cair, reabra e re-execute desde a Célula 1.

models_str = ' '.join(LSSVM_MODELS)
datasets_str = 'BCW PID HAB VCP GCR AUS TWS TWM TWC'

!python -u scripts/run_tier1_gridcv.py \
    --models {models_str} \
    --datasets {datasets_str} \
    --output results/tier1_gridcv_lssvm.json \
    --log-level INFO 2>&1 | tee /tmp/tier1_lssvm_run.log

In [ ]:
# ── Célula 10: Save final no Drive ──────────────────────────────────────────
import shutil, signal
from pathlib import Path

try:
    sync_proc.send_signal(signal.SIGTERM)
except Exception:
    pass

drive_dest = Path(DRIVE_PATH)
src = Path('results') / OUTPUT_FNAME
dst = drive_dest / OUTPUT_FNAME
if src.exists():
    shutil.copy(src, dst)
    print(f'✓ Salvo: {dst.name} ({src.stat().st_size / 1024:.1f} KB)')

print(f'\nConteúdo do Drive:')
!ls -lh '{DRIVE_PATH}'

In [ ]:
# ── Célula 11: Resumo dos resultados ────────────────────────────────────────
import json
from collections import defaultdict
import statistics as st

records = json.load(open('results/' + OUTPUT_FNAME))
print(f'Total records: {len(records)}\n')

agg = defaultdict(lambda: defaultdict(list))
for r in records:
    if r.get('status') != 'ok':
        continue
    agg[r['variant']][r['dataset']].append(r['test_f1_macro'])

datasets = 'BCW PID HAB VCP GCR AUS TWS TWM TWC'.split()
header = ['Modelo'] + datasets + ['Média']
print(f'{header[0]:<22}' + ''.join(f'{h:>7}' for h in header[1:]))
print('-' * (22 + 7 * len(header[1:])))
for variant in LSSVM_MODELS:
    line = [variant]
    means = []
    for d in datasets:
        vals = agg[variant][d]
        if vals:
            m = st.mean(vals)
            line.append(f'{m:>7.3f}')
            means.append(m)
        else:
            line.append(f'{"-":>7}')
    avg = st.mean(means) if means else 0
    line.append(f'{avg:>7.3f}')
    print(f'{line[0]:<22}' + ''.join(line[1:]))